# Quick UrQMD ROOT plots

Run this notebook in the `urqmd` environment. It auto-finds a ROOT file and makes quick sanity plots.

In [1]:
import ROOT
from pathlib import Path

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(1110)
ROOT.gStyle.SetPalette(ROOT.kViridis)

plots_dir = Path("../plots").resolve()
plots_dir.mkdir(exist_ok=True)

search_dirs = [
    Path("/research/Data"),
    Path("/workspace/research/Data"),
    Path("/workspace/Data"),
]
preferred = Path("/research/Data/combined_100k.root")
mounted = Path("/workspace/research/Data/combined_100k.root")

if preferred.is_file():
    root_path = preferred
elif mounted.is_file():
    root_path = mounted
else:
    files = [p for d in search_dirs if d.is_dir() for p in d.rglob("*.root")]
    if not files:
        raise FileNotFoundError(f"No ROOT files found under {search_dirs}")
    root_path = max(files, key=lambda p: ("combined" in p.name.lower(), p.stat().st_size))

f = ROOT.TFile.Open(str(root_path))
if not f or f.IsZombie():
    raise OSError(f"Could not open ROOT file: {root_path}")

tree = f.Get("urqmd")
if not tree:
    f.ls()
    raise KeyError("TTree 'urqmd' was not found; see f.ls() above")

branches = [b.GetName() for b in tree.GetListOfBranches()]
print(f"File: {root_path}")
print(f"Entries: {tree.GetEntries():,}")
print("Branches:", ", ".join(branches))


File: /workspace/research/Data/combined_100k.root
Entries: 100,000
Branches: mul, b, Npart, pid, px, py, pz


In [2]:
# Impact parameter b
c_b = ROOT.TCanvas("c_b", "Impact parameter", 900, 650)
h_b = ROOT.TH1F("h_b", "UrQMD impact parameter;Impact parameter b [fm];Events", 100, 0, 20)
tree.Draw("b>>h_b")
h_b.SetLineWidth(2)
h_b.SetFillColorAlpha(ROOT.kAzure - 9, 0.55)
c_b.SetGrid()
# c_b.SetLogy(True)  # Uncomment for log scale
# c_b.SaveAs(str(plots_dir / "impact_parameter_b.png"))
c_b.Draw()
c_b


In [3]:
# Number of participant nucleons
c_npart = ROOT.TCanvas("c_npart", "Npart", 900, 650)
h_npart = ROOT.TH1F("h_npart", "UrQMD participant nucleons;N_{part};Events", 100, 0, 450)
tree.Draw("Npart>>h_npart")
h_npart.SetLineWidth(2)
h_npart.SetFillColorAlpha(ROOT.kOrange - 2, 0.55)
c_npart.SetGrid()
# c_npart.SetLogy(True)  # Uncomment for log scale
# c_npart.SaveAs(str(plots_dir / "npart.png"))
c_npart.Draw()
c_npart


In [4]:
# Event multiplicity
c_mul = ROOT.TCanvas("c_mul", "Multiplicity", 900, 650)
h_mul = ROOT.TH1F("h_mul", "UrQMD event multiplicity;Event multiplicity;Events", 120, 0, 3000)
tree.Draw("mul>>h_mul")
h_mul.SetLineWidth(2)
h_mul.SetFillColorAlpha(ROOT.kGreen + 1, 0.55)
c_mul.SetGrid()
# c_mul.SetLogy(True)  # Uncomment for log scale
# c_mul.SaveAs(str(plots_dir / "event_multiplicity.png"))
c_mul.Draw()
c_mul


In [5]:
# 2D: Npart vs impact parameter
c_npart_b = ROOT.TCanvas("c_npart_b", "Npart vs b", 950, 700)
h_npart_b = ROOT.TH2F("h_npart_b", "N_{part} vs impact parameter;Impact parameter b [fm];N_{part}", 100, 0, 20, 100, 0, 450)
tree.Draw("Npart:b>>h_npart_b", "", "COLZ")
c_npart_b.SetRightMargin(0.15)
c_npart_b.SetGrid()
# c_npart_b.SetLogz(True)  # Uncomment for log color scale
# c_npart_b.SaveAs(str(plots_dir / "npart_vs_b.png"))
c_npart_b.Draw()
c_npart_b


In [6]:
# 2D: multiplicity vs impact parameter
c_mul_b = ROOT.TCanvas("c_mul_b", "Multiplicity vs b", 950, 700)
h_mul_b = ROOT.TH2F("h_mul_b", "Multiplicity vs impact parameter;Impact parameter b [fm];Event multiplicity", 100, 0, 20, 120, 0, 3000)
tree.Draw("mul:b>>h_mul_b", "", "COLZ")
c_mul_b.SetRightMargin(0.15)
c_mul_b.SetGrid()
# c_mul_b.SetLogz(True)  # Uncomment for log color scale
# c_mul_b.SaveAs(str(plots_dir / "multiplicity_vs_b.png"))
c_mul_b.Draw()
c_mul_b


In [7]:
# Particle PID distribution
c_pid = ROOT.TCanvas("c_pid", "Particle PID", 900, 650)
h_pid = ROOT.TH1F("h_pid", "UrQMD particle PID distribution;PDG particle ID;Particles", 5000, -2500, 2500)
tree.Draw("pid>>h_pid")
h_pid.SetLineWidth(2)
h_pid.SetFillColorAlpha(ROOT.kMagenta - 7, 0.55)
c_pid.SetLogy(True)  # Change to False for linear scale
c_pid.SetGrid()
# c_pid.SaveAs(str(plots_dir / "particle_pid.png"))
c_pid.Draw()
c_pid


In [8]:
# Particle transverse momentum
c_pt = ROOT.TCanvas("c_pt", "Particle pT", 900, 650)
h_pt = ROOT.TH1F("h_pt", "UrQMD particle transverse momentum;p_{T} [GeV/c];Particles", 120, 0, 3)
tree.Draw("sqrt(px*px+py*py)>>h_pt")
h_pt.SetLineWidth(2)
h_pt.SetFillColorAlpha(ROOT.kCyan - 3, 0.55)
c_pt.SetLogy(True)  # Change to False for linear scale
c_pt.SetGrid()
# c_pt.SaveAs(str(plots_dir / "particle_pt.png"))
c_pt.Draw()
c_pt


In [9]:
# Proton transverse momentum
c_proton_pt = ROOT.TCanvas("c_proton_pt", "Proton pT", 900, 650)
h_proton_pt = ROOT.TH1F("h_proton_pt", "UrQMD proton transverse momentum;p_{T} [GeV/c];Protons", 120, 0, 3)
tree.Draw("sqrt(px*px+py*py)>>h_proton_pt", "pid==2212")
h_proton_pt.SetLineWidth(2)
h_proton_pt.SetFillColorAlpha(ROOT.kRed - 7, 0.55)
c_proton_pt.SetLogy(True)  # Change to False for linear scale
c_proton_pt.SetGrid()
# c_proton_pt.SaveAs(str(plots_dir / "proton_pt.png"))
c_proton_pt.Draw()
c_proton_pt


In [10]:
# Proton longitudinal momentum
c_proton_pz = ROOT.TCanvas("c_proton_pz", "Proton pz", 900, 650)
h_proton_pz = ROOT.TH1F("h_proton_pz", "UrQMD proton longitudinal momentum;p_{z} [GeV/c];Protons", 160, -10, 10)
tree.Draw("pz>>h_proton_pz", "pid==2212")
h_proton_pz.SetLineWidth(2)
h_proton_pz.SetFillColorAlpha(ROOT.kViolet - 6, 0.55)
c_proton_pz.SetLogy(True)  # Change to False for linear scale
c_proton_pz.SetGrid()
# c_proton_pz.SaveAs(str(plots_dir / "proton_pz.png"))
c_proton_pz.Draw()
c_proton_pz
